# 05 – Character Anime Works

Esplorazione e data cleaning del dataset `character_anime_works.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `anime_mal_id` | ID univoco dell'anime su MyAnimeList |
| `character_mal_id` | ID univoco del personaggio su MyAnimeList |
| `character_name` | Nome del personaggio |
| `role` | Ruolo del personaggio nell'anime |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_works = pd.read_csv('../datasets/character_anime_works.csv')
print(f'Shape: {df_works.shape}')
print()
df_works.info()
print()
df_works.head()

Shape: (236816, 4)

<class 'pandas.DataFrame'>
RangeIndex: 236816 entries, 0 to 236815
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   anime_mal_id      236816 non-null  int64
 1   character_mal_id  236816 non-null  int64
 2   character_name    236816 non-null  str  
 3   role              236816 non-null  str  
dtypes: int64(2), str(2)
memory usage: 7.2 MB



,anime_mal_id,character_mal_id,character_name,role
0,2928,5781,Atoli,Main
1,2928,33,Haseo,Main
2,2928,32,Ovan,Main
3,2928,34,Shino,Main
4,2928,5785,Aina,Supporting


Il dataset contiene **236.816 righe** e **4 colonne**. Tutte le colonne sono complete: **nessun valore nullo** in nessuna delle 4 colonne. I tipi di dati sono adeguati: `int64` per gli ID e `str` per le colonne testuali.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [2]:
n_originale = len(df_works)

mask_dup = df_works.duplicated(keep=False)       # tutte le occorrenze coinvolte
n_righe_coinvolte = mask_dup.sum()                    # righe totali che hanno almeno un duplicato
n_gruppi = df_works[mask_dup].duplicated(keep='first').sum()  # occorrenze extra (da rimuovere)
n_tenute = n_righe_coinvolte - n_gruppi               # prime occorrenze mantenute

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_works.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_works):,}')

Righe totali coinvolte in duplicazioni : 0
  → prime occorrenze mantenute         : 0
  → occorrenze extra rimosse           : 0

Righe prima della rimozione : 236,816
Righe dopo la rimozione     : 236,816


Nessun duplicato esatto trovato. Tutte le 236.816 righe sono già uniche. Il dataset rimane invariato.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `anime_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `mal_id` di `details.csv`.

I valori duplicati sono **attesi**: ogni anime è associato a più personaggi.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `details_clean.csv`.

Usiamo `check_fk` per entrambi i controlli.

In [3]:
df_details = pd.read_csv('../datasets_cleaned/details_clean.csv')

mask_orphan_anime = check_fk(df_works['anime_mal_id'], df_details['mal_id'], child_df=df_works)

print(f'Null in anime_mal_id            : {df_works["anime_mal_id"].isna().sum()}')
print(f'Duplicati in anime_mal_id (attesi): {df_works["anime_mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        anime_mal_id
  Colonna PK  (tabella padre)         mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       236,816
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            236,816  (100.00%)
  Valori unici nella PK padre         28,955

  ✓  Righe con FK valida              236,816  (100.00%)
  ✗  Righe orfane (FK non in PK)      0  (0.00%)
     → ID orfani unici                0

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

Nessuna violazione. Tutti i valori FK esistono nella tabella padre.

Null in anime_mal_id            : 0
Duplicati in anime_mal_id (attesi): 221,532


**Osservazioni:**
- **Nessun valore nullo**: tutti i 236.816 record hanno un ID anime valido.
- **Integrità referenziale**: nessuna riga orfana. Tutti i 15.284 ID anime presenti in questo dataset esistono in `details_clean.csv`.
- **Duplicati attesi**: ogni anime è associato a più personaggi.

**Nessuna pulizia è necessaria.**